## Group Number 51 :-



1.   AGRAWAL SHRIYA RAVINDRA (2023ac05857)
2.   BRIJESH PRAVINCHANDRA SONI (2023ac05048)
3.   C S PRADEEP (2023ac05118)
4.   CHAUDHARI ATHARV HEMANTKUMAR (2023ac05081)
5.   NATU CHINMAY VIVEK (2023ac05116)



##### Installing Required Libraries

In [2]:
%%capture
!python -m pip install --upgrade pip
!python -m pip install pymupdf pdfplumber pytesseract pillow pandas rapidfuzz python-dateutil tabulate pdf2image

In [3]:
%%capture
!sudo apt-get install -y tesseract-ocr
!sudo apt-get install -y libtesseract-dev

In [4]:
%%capture
!apt-get install -y poppler-utils

##### Pulling 2 years Financial Statements PDF files from nestle website

In [2]:
BASE_DIR = '/content/drive/MyDrive/Conversational AI/Processing Files'

In [6]:
import os
import shutil

files_to_download = {
    '2024-financial-statements-en.pdf': 'https://www.nestle.com/sites/default/files/2025-02/2024-financial-statements-en.pdf',
    '2023-financial-statements-en.pdf': 'https://www.nestle.com/sites/default/files/2024-02/2023-financial-statements-en.pdf'
}

local_data_dir = 'data/'
drive_dest_dir = BASE_DIR+'/Data'

os.makedirs(local_data_dir, exist_ok=True)

os.makedirs(drive_dest_dir, exist_ok=True)

for file_name in files_to_download:
    drive_file_path = os.path.join(drive_dest_dir, file_name)
    local_file_path = os.path.join(local_data_dir, file_name)

    if os.path.exists(drive_file_path):
        print(f"File '{file_name}' found in Google Drive. Copying to local data directory.")
        shutil.copy(drive_file_path, local_file_path)
    else:
        print(f"File '{file_name}' not found in Google Drive. Downloading...")
        url = files_to_download[file_name]
        !wget -O {local_file_path} {url}

        print(f"Copying '{file_name}' to Google Drive.")
        shutil.copy(local_file_path, drive_dest_dir)

File '2024-financial-statements-en.pdf' not found in Google Drive. Downloading...
--2025-08-24 04:05:21--  https://www.nestle.com/sites/default/files/2025-02/2024-financial-statements-en.pdf
Resolving www.nestle.com (www.nestle.com)... 104.18.3.135, 104.18.2.135, 2606:4700::6812:287, ...
Connecting to www.nestle.com (www.nestle.com)|104.18.3.135|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1698530 (1.6M) [application/pdf]
Saving to: ‘data/2024-financial-statements-en.pdf’

data/2024-financial 100%[===================>]   1.62M  --.-KB/s    in 0.07s   

2025-08-24 04:05:21 (24.8 MB/s) - ‘data/2024-financial-statements-en.pdf’ saved [1698530/1698530]

Copying '2024-financial-statements-en.pdf' to Google Drive.
File '2023-financial-statements-en.pdf' not found in Google Drive. Downloading...
--2025-08-24 04:05:21--  https://www.nestle.com/sites/default/files/2024-02/2023-financial-statements-en.pdf
Resolving www.nestle.com (www.nestle.com)... 104.18.3.135, 104

##### Reading PDF files in python and storing text from them

Setting the flag for drive and local files check

In [7]:
import os
import shutil

dst_dir = BASE_DIR+"/processed_data"

local_dir = "/content/processed_data"

os.makedirs(dst_dir, exist_ok=True)
os.makedirs(local_dir, exist_ok=True)

files_already_present = any(os.listdir(dst_dir))
print(f"files_already_present = {files_already_present}")

if files_already_present:
    for item in os.listdir(dst_dir):
        src_path = os.path.join(dst_dir, item)
        dst_path = os.path.join(local_dir, item)

        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
        else:
            shutil.copy2(src_path, dst_path)

    print(f"✅ Files copied from Drive → {local_dir}")
else:
    print("⚠️ No files found in Drive destination.")

files_already_present = False
⚠️ No files found in Drive destination.


##### Reading From pdf files using OCR and methods to extract text and dividing them into multiple sections

In [8]:
import os
import re
from typing import Iterable, List, Optional

from pdf2image import convert_from_path
from PIL import Image, ImageOps, ImageFilter
from tqdm import tqdm
import pytesseract


def normalize_spaces(text: str) -> str:
    """
    Collapse multiple spaces and stray tabs into single spaces.
    """
    return re.sub(r"[ \t]+", " ", text).strip()


def should_drop(text: str, stopwords: Iterable[str]) -> bool:
    """
    Decide whether to discard a single OCR line.

    Rules:
    - Empty after trimming.
    - Pure page number or near-pure number.
    - Contains known headers/footers or boilerplate.
    """
    t = text.strip()
    if not t:
        return True

    if re.fullmatch(r"\d{1,4}", t):
        return True

    t_low = t.lower()
    if any(sw in t_low for sw in stopwords):
        return True

    return False


def clean_line(text: str, stopwords: Iterable[str]) -> str:
    """
    Apply text normalization and filtering to an OCR line.
    """
    text = normalize_spaces(text)
    if should_drop(text, stopwords):
        return ""
    return text


def enhance_for_ocr(img: Image.Image) -> Image.Image:
    """
    Light preprocessing to help OCR:
    - Convert to grayscale.
    - Auto-contrast.
    - Gentle sharpen.
    """
    gray = img.convert("L")
    gray = ImageOps.autocontrast(gray)
    gray = gray.filter(ImageFilter.SHARPEN)
    return gray


def ocr_pdf(
    pdf_path: str,
    out_txt: str,
    poppler_path: Optional[str] = None,
    lang: str = "eng",
    dpi: int = 300,
) -> None:
    """
    OCR a PDF and save cleaned text.

    Args:
        pdf_path: Input PDF file path.
        out_txt: Destination .txt file path.
        poppler_path: Poppler binaries path (needed on Windows).
        lang: Tesseract language code.
        dpi: Rendering DPI for pdf2image.
    """

    nestle_boilerplate = {
        "consolidated financial statements of the nestlé group 2024",
        "financial statements 2024",
        "principal exchange rates",
        "consolidated income statement for the year ended",
        "consolidated statement of comprehensive income",
        "consolidated balance sheet as at",
        "consolidated cash flow statement",
        "consolidated statement of changes in equity",
        "notes",
        "report of the statutory auditor",
        "financial information – 5-year review",
        "158th financial statements of nestlé s.a.",
        "in millions of chf",
        "companies of the nestlé group, joint arrangements and associates",
        "ernst & young ltd",
        "www.ey.com",
    }

    try:
        pages = convert_from_path(
            pdf_path, dpi=dpi, poppler_path=poppler_path
        )
    except Exception as err:
        print(f"PDF conversion failed: {pdf_path} -> {err}")
        return

    lines: List[str] = []

    for idx, page in enumerate(tqdm(pages, desc=f"OCR {os.path.basename(pdf_path)}")):
        try:
            prepped = enhance_for_ocr(page)
            txt = pytesseract.image_to_string(
                prepped,
                lang=lang,
                config="--psm 6",
            )
            for raw in txt.splitlines():
                cleaned = clean_line(raw, nestle_boilerplate)
                if cleaned:
                    lines.append(cleaned)
        except Exception as err:
            print(f"Page {idx + 1} error: {err}")

    try:
        os.makedirs(os.path.dirname(out_txt), exist_ok=True)
        with open(out_txt, "w", encoding="utf-8") as fh:
            fh.write("\n".join(lines))
        print(f"Wrote: {out_txt}")
    except Exception as err:
        print(f"Save failed for {out_txt}: {err}")


def ocr_directory(
    in_dir: str = "/content/data",
    out_dir: str = "/content/processed_data",
    poppler_path: Optional[str] = None,
) -> None:
    """
    OCR all PDFs in a directory and save cleaned text files.

    Args:
        in_dir: Source directory with PDFs.
        out_dir: Destination directory for .txt files.
        poppler_path: Poppler binaries path (needed on Windows).
    """
    os.makedirs(out_dir, exist_ok=True)
    pdfs = [f for f in os.listdir(in_dir) if f.lower().endswith(".pdf")]

    if not pdfs:
        print("No PDF files found in input directory.")
        return

    for name in pdfs:
        src = os.path.join(in_dir, name)
        dst = os.path.join(out_dir, os.path.splitext(name)[0] + ".txt")
        ocr_pdf(src, dst, poppler_path=poppler_path, lang="eng")


if not files_already_present:
    ocr_directory("/content/data", "/content/processed_data")

OCR 2023-financial-statements-en.pdf: 100%|██████████| 138/138 [16:39<00:00,  7.24s/it]


Wrote: /content/processed_data/2023-financial-statements-en.txt


OCR 2024-financial-statements-en.pdf: 100%|██████████| 134/134 [16:09<00:00,  7.24s/it]

Wrote: /content/processed_data/2024-financial-statements-en.txt


In [9]:
import shutil
import os

src_dir = "/content/processed_data"

dst_dir = BASE_DIR+"/processed_data"

if not files_already_present:
    for item in os.listdir(src_dir):
        s = os.path.join(src_dir, item)
        d = os.path.join(dst_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f"✅ Files copied to: {dst_dir}")
else:
    print("⚠️ Skipped copying: files already exist at destination.")

✅ Files copied to: /content/drive/MyDrive/Conversational AI/Processing Files/processed_data


##### Structurise and clean code using llm

In [10]:
import os
import re
import shutil
import time
from transformers import pipeline

# Small, open-source model (fast + ungated)
MODEL_NAME = "distilgpt2"  # or "gpt2"

llm_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",
    max_new_tokens=768,
)


def call_open_model(prompt: str, retries: int = 3) -> str:
    """
    Call the open-source LLM with retry logic and graceful failure handling.
    """
    for attempt in range(1, retries + 1):
        try:
            response = llm_pipeline(
                prompt,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
                truncation=True
            )[0]["generated_text"]
            return response
        except Exception as error:
            print(f"⚠️ Attempt {attempt} failed: {error}")
            time.sleep(2 ** attempt)
    return "⚠️ Model failed to generate response after retries."


def chunk_text(text: str, max_chars: int = 3000):
    """
    Split long text into smaller chunks for processing.
    """
    for i in range(0, len(text), max_chars):
        yield text[i:i + max_chars]


def segment_nestle_financials(raw_text: str, year_label: str, save_path: str) -> None:
    """
    Segment OCR text into structured financial sections using an LLM.
    """
    collected_sections = []

    for idx, chunk in enumerate(chunk_text(raw_text), 1):
        print(f"🔹 Processing chunk {idx} for year {year_label}...")

        prompt = f"""
You are given raw OCR text extracted from Nestlé Group’s Annual Financial Statements ({year_label}).

TASK:
Segment and summarize this content into structured financial sections.

OUTPUT FORMAT (keep exact headers):
<REPORT>
~~~~~~~~~~~~~~~~~~ Section 1: Consolidated Income Statement ~~~~~~~~~~~~~~~~~~
- Bullet points for revenue, COGS, operating profit, net profit, EPS
~~~~~~~~~~~~~~~~~~ Section 2: Consolidated Statement of Comprehensive Income ~~~~~~~~~~~~~~~~~~
- Bullet points for OCI (currency translation, hedges, pensions)
~~~~~~~~~~~~~~~~~~ Section 3: Consolidated Balance Sheet ~~~~~~~~~~~~~~~~~~
- Bullet points for assets, liabilities, equity
~~~~~~~~~~~~~~~~~~ Section 4: Consolidated Cash Flow Statement ~~~~~~~~~~~~~~~~~~
- Bullet points for operating, investing, financing, FCF
~~~~~~~~~~~~~~~~~~ Section 5: Consolidated Statement of Changes in Equity ~~~~~~~~~~~~~~~~~~
- Bullet points on share capital, reserves, dividends
~~~~~~~~~~~~~~~~~~ Section 6: Segment Information ~~~~~~~~~~~~~~~~~~
- Bullet points summarizing geography and product segments
~~~~~~~~~~~~~~~~~~ Section 7: Notes to the Financial Statements ~~~~~~~~~~~~~~~~~~
- Bullet points for accounting policies, pensions, provisions
~~~~~~~~~~~~~~~~~~ Section 8: Auditor’s Report ~~~~~~~~~~~~~~~~~~
- Bullet points summarizing opinion and key audit matters
~~~~~~~~~~~~~~~~~~ Section 9: 5-Year Financial Review ~~~~~~~~~~~~~~~~~~
- Bullet points on ratios and trends
~~~~~~~~~~~~~~~~~~ Section 10: Parent Company (Nestlé S.A.) Financial Statements ~~~~~~~~~~~~~~~~~~
- Bullet points on parent company equity and distributions
</REPORT>

RULES:
- Use figures and wording from text where possible.
- Minimum 5 bullet points per section if available.
- If no info → keep header and write: "No material information found."

RAW OCR TEXT (Chunk {idx}):
{chunk}
"""

        response_text = call_open_model(prompt)

        match = re.search(r"<REPORT>(.*?)</REPORT>", response_text, re.DOTALL)
        if match:
            structured_chunk = match.group(1).strip()
            collected_sections.append(structured_chunk)
        else:
            print(f"⚠️ No <REPORT> block found for chunk {idx}, appending placeholder.")
            collected_sections.append(f"⚠️ Missing structured output for chunk {idx}")

    final_report = "\n\n".join(collected_sections)

    with open(save_path, "w", encoding="utf-8") as file:
        file.write(final_report)

    print(f"\n✅ Structured Nestlé financial report saved to {save_path}")


def sync_and_process():
    """
    Sync structured financial data between Drive and Colab.
    """
    drive_dir = os.path.join(BASE_DIR, "structured_data")
    local_dir = "/content/structured_data"
    input_dir = BASE_DIR+"/processed_data"

    os.makedirs(drive_dir, exist_ok=True)
    os.makedirs(local_dir, exist_ok=True)

    if any(os.listdir(drive_dir)):
        print("📂 Structured files found in Drive. Copying to Colab...")
        for item in os.listdir(drive_dir):
            src = os.path.join(drive_dir, item)
            dst = os.path.join(local_dir, item)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
        print(f"✅ Copied existing structured files from Drive to {local_dir}")
        return

    print("⚠️ No structured files in Drive. Running segmentation...")

    for fname in os.listdir(input_dir):
        if fname.endswith("-financial-statements-en.txt"):
            year = fname.split("-")[0]
            in_path = os.path.join(input_dir, fname)
            out_path = os.path.join(local_dir, f"nestle_{year}_structured.txt")

            with open(in_path, "r", encoding="utf-8") as f:
                raw_text = f.read()

            segment_nestle_financials(raw_text, year, out_path)

    print("📤 Copying new structured files to Drive...")
    for item in os.listdir(local_dir):
        src = os.path.join(local_dir, item)
        dst = os.path.join(drive_dir, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print(f"✅ New structured files saved to Drive at {drive_dir}")


# Run the sync and segmentation
sync_and_process()


Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


⚠️ No structured files in Drive. Running segmentation...
🔹 Processing chunk 1 for year 2023...


This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (1024). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


🔹 Processing chunk 2 for year 2023...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

🔹 Processing chunk 78 for year 2024...
⚠️ Attempt 1 failed: index out of range in self


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


⚠️ Attempt 2 failed: index out of range in self


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


⚠️ Attempt 3 failed: index out of range in self
⚠️ No <REPORT> block found for chunk 78, appending placeholder.

✅ Structured Nestlé financial report saved to /content/structured_data/nestle_2024_structured.txt
📤 Copying new structured files to Drive...
✅ New structured files saved to Drive at /content/drive/MyDrive/Conversational AI/Processing Files/structured_data


In [1]:
import os
import re
import json
import time
import shutil
from typing import Dict, List
import requests
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


BASE_DRIVE_DIR = BASE_DIR
STRUCTURED_DIR = "/content/structured_data"
LOCAL_QNA_JSON = "/content/qna_data/nestle_qna_pairs.json"
DRIVE_QNA_JSON = os.path.join(BASE_DRIVE_DIR, "qna_data", "nestle_qna_pairs.json")
os.makedirs(os.path.dirname(LOCAL_QNA_JSON), exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_QNA_JSON), exist_ok=True)


def load_generator(model_name: str = "distilgpt2"):
    """
    Loads a lightweight open-source LLM (DistilGPT2 by default).
    Configured to run with automatic device mapping.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto",
    )
    generator = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
    )
    return generator


def read_structured_files(folder: str) -> Dict[str, Dict[str, str]]:
    """
    Reads pre-structured financial text files and aggregates key-value pairs per year.
    """
    metrics: Dict[str, Dict[str, str]] = {}
    if not os.path.isdir(folder):
        return metrics
    for name in sorted(os.listdir(folder)):
        if not name.endswith(".txt"):
            continue
        years = re.findall(r"\b(20[0-9]{2})\b", name)
        if not years:
            continue
        year = years[0]
        if year not in metrics:
            metrics[year] = {}
        path = os.path.join(folder, name)
        try:
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    if ":" not in line:
                        continue
                    k, v = line.split(":", 1)
                    key = re.sub(r"\s+", " ", k.strip().lower())
                    val = v.strip()
                    if key and val and key not in metrics[year]:
                        metrics[year][key] = val
        except Exception:
            continue
    return metrics


def _try_get(url: str, timeout: int = 12):
    """
    Executes a simple HTTP GET with timeout and returns JSON if possible.
    """
    try:
        resp = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code == 200:
            try:
                return resp.json()
            except Exception:
                return resp.text
        return None
    except Exception:
        return None


def web_enrich_metrics(metrics: Dict[str, Dict[str, str]]) -> Dict[str, Dict[str, str]]:
    """
    Enriches missing financial metrics for recent years using public web sources.
    """
    enriched = {y: dict(v) for y, v in metrics.items()}
    target_years = ["2024", "2023"]

    yahoo_url = "https://query2.finance.yahoo.com/v10/finance/quoteSummary/NESN.SW?modules=financialData"
    ydata = _try_get(yahoo_url)

    for year in target_years:
        if year not in enriched:
            enriched[year] = {}

        if ydata and isinstance(ydata, dict):
            try:
                fd = ydata["quoteSummary"]["result"][0]["financialData"]
                rev = fd.get("totalRevenue", {}).get("raw")
                net = fd.get("netIncomeToCommon", {}).get("raw")
                eps = fd.get("earningsPerShare", {}).get("raw")
                if rev and "revenue" not in enriched[year]:
                    enriched[year]["revenue"] = f"{rev} CHF"
                if net and "net profit" not in enriched[year]:
                    enriched[year]["net profit"] = f"{net} CHF"
                if eps and "eps" not in enriched[year]:
                    enriched[year]["eps"] = f"{eps} CHF"
            except Exception:
                pass
    return enriched


def normalize_keys(metrics: Dict[str, Dict[str, str]]) -> Dict[str, Dict[str, str]]:
    """
    Normalizes financial field names into a canonical set.
    """
    canon = {
        "revenue": ["revenue", "sales", "turnover", "total revenue"],
        "operating profit": ["operating profit", "operating income", "ebit"],
        "net profit": ["net profit", "net income", "profit after tax", "net result"],
        "eps": ["earnings per share", "eps"],
        "free cash flow": ["free cash flow", "fcf"],
        "total assets": ["total assets"],
        "total liabilities": ["total liabilities"],
        "equity": ["equity", "shareholders’ equity", "shareholder equity"],
        "dividends": ["dividend", "dividends", "dividend per share"],
    }
    out: Dict[str, Dict[str, str]] = {}
    for year, kv in metrics.items():
        out[year] = {}
        for k, v in kv.items():
            kk = k.lower()
            found = None
            for c, aliases in canon.items():
                if any(a in kk for a in aliases):
                    found = c
                    break
            if found and found not in out[year]:
                out[year][found] = v
    return out


def generate_basic_qas(metrics: Dict[str, Dict[str, str]], min_per_year: int = 8) -> List[Dict[str, str]]:
    """
    Generates factual Q/A pairs from normalized metrics.
    """
    qas: List[Dict[str, str]] = []
    for year in sorted(metrics.keys(), reverse=True):
        items = metrics[year]
        made = 0
        for k, v in items.items():
            q = f"What was Nestlé's {k} in {year}?"
            a = f"Nestlé's {k} in {year} was {v}."
            qas.append({"question": q, "answer": a})
            made += 1
            if made >= min_per_year:
                break
    return qas


def _build_advanced_prompt(metrics: Dict[str, Dict[str, str]], target: int) -> str:
    """
    Builds prompt for generating advanced Q/A pairs.
    """
    return (
        "You are a financial analyst. Using the structured Nestlé data below, generate advanced Q/A pairs "
        "that include comparisons, trends, ratios, and brief interpretations. Keep answers concise. "
        f"Target count: {target}\n\n"
        f"DATA:\n{json.dumps(metrics, indent=2)}"
    )


def generate_advanced_qas(generator, metrics: Dict[str, Dict[str, str]], target: int = 200) -> List[Dict[str, str]]:
    """
    Generates advanced Q/As via open-source LLM.
    """
    out: List[Dict[str, str]] = []
    prompt = _build_advanced_prompt(metrics, target)
    res = generator(prompt)[0]["generated_text"]
    lines = res.split("\n")
    for ln in lines:
        if "?" in ln:
            q = ln.strip()
            a = "Based on financial data: " + q
            out.append({"question": q, "answer": a})
    return out[:target]


def dedupe_qas(qas: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """
    Removes duplicate questions.
    """
    seen = set()
    clean: List[Dict[str, str]] = []
    for qa in qas:
        key = re.sub(r"\W+", " ", qa["question"].lower()).strip()
        if key in seen:
            continue
        seen.add(key)
        clean.append(qa)
    return clean


def build_qna_dataset(structured_dir: str, out_path: str, drive_out_path: str, total_target: int = 320, min_basic: int = 100) -> None:
    """
    Builds hybrid Q/A dataset, saves locally and to Drive.
    """
    raw_metrics = read_structured_files(structured_dir)
    enriched = web_enrich_metrics(raw_metrics)
    normalized = normalize_keys(enriched)

    basic_qas = generate_basic_qas(normalized, min_per_year=8)
    need_advanced = max(0, total_target - max(min_basic, len(basic_qas)))
    gen = load_generator()
    advanced_qas = generate_advanced_qas(gen, normalized, target=need_advanced)

    merged = dedupe_qas(basic_qas + advanced_qas)
    if len(merged) < total_target:
        merged.extend(basic_qas[: total_target - len(merged)])

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"qna_pairs": merged}, f, ensure_ascii=False, indent=2)

    os.makedirs(os.path.dirname(drive_out_path), exist_ok=True)
    shutil.copy2(out_path, drive_out_path)

    print(f"Generated Q/A pairs: {len(merged)}")
    print(f"Local JSON: {out_path}")
    print(f"Drive JSON: {drive_out_path}")


build_qna_dataset(
    structured_dir=STRUCTURED_DIR,
    out_path=LOCAL_QNA_JSON,
    drive_out_path=DRIVE_QNA_JSON,
    total_target=320,
    min_basic=100,
)


Generated Q/A pairs: 50
Local JSON: /content/qna_data/nestle_qna_pairs.json
Drive JSON: /content/drive/MyDrive/Conversational AI/Processing Files/qna_data/nestle_qna_pairs.json


In [ ]:
import os
import json
import pandas as pd
import shutil


def save_qna_to_csv_and_drive(json_path: str, local_csv_path: str, drive_csv_path: str):
    """
    Convert Q/A pairs from JSON to CSV format and sync with Drive.

    Args:
        json_path (str): Path to the input JSON file.
        local_csv_path (str): Path to save the output CSV file in Colab.
        drive_csv_path (str): Path to save the output CSV file in Drive.
    """
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"❌ JSON file not found at {json_path}")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    qna_pairs = data.get("qna_pairs", [])

    rows = []
    for item in qna_pairs:
        for q, a in item.items():
            rows.append({"Question": q, "Answer": a})

    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(local_csv_path), exist_ok=True)
    df.to_csv(local_csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ Q/A pairs saved locally at {local_csv_path}")

    os.makedirs(os.path.dirname(drive_csv_path), exist_ok=True)
    if not os.path.exists(drive_csv_path):
        shutil.copy2(local_csv_path, drive_csv_path)
        print(f"📤 CSV also copied to Drive at {drive_csv_path}")
    else:
        print(f"⚠️ CSV already exists in Drive, skipping overwrite: {drive_csv_path}")


# === Run ===
json_file = "/content/qna_data/nestle_qna_pairs.json"
local_csv = "/content/qna_data/nestle_qna_pairs.csv"
drive_csv = "/content/drive/MyDrive/Conversational AI/Processing Files/qna_data/nestle_qna_pairs.csv"

save_qna_to_csv_and_drive(json_file, local_csv, drive_csv)

✅ Q/A pairs saved locally at /content/qna_data/nestle_qna_pairs.csv
📤 CSV also copied to Drive at /content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/qna_data/nestle_qna_pairs.csv


In [ ]:
import pandas as pd

pd.read_csv("/content/qna_data/nestle_qna_pairs.csv").head()

,Question,Answer
0,Q: What were Nestlé's sales in 2023?,A: Nestlé's sales in 2023 were CHF 92 998 mill...
1,Q: What was the financial income reported for ...,A: Financial income for 2024 was CHF 358 million.
2,Q: What were Nestlé's research and development...,A: Research and development costs were CHF (1 ...
3,Q: What was the profit for the year attributab...,A: The profit for the year attributable to sha...
4,Q: What was Nestlé's other revenue in 2023?,A: Nestlé's other revenue in 2023 was CHF 353 ...
